In [1]:
# Clone your team repo
!git clone https://github.com/Harshithk09/ECS-170-Group-project.git
%cd ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template

!pip install -q scikit-learn

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | device: {device}')
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Cloning into 'ECS-170-Group-project'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (449/449), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 449 (delta 216), reused 426 (delta 193), pack-reused 0 (from 0)
Receiving objects: 100% (449/449), 4.00 MiB | 6.99 MiB/s, done.
Resolving deltas: 100% (216/216), done.
/content/ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template
PyTorch 2.11.0+cu128 | device: cuda
GPU: Tesla T4


In [2]:
# Upload stage_5_data.zip and extract into data/stage_5_data/
from google.colab import files
import zipfile, os

uploaded = files.upload()   # select stage_5_data.zip
zip_name = list(uploaded.keys())[0]

os.makedirs('data/stage_5_data', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('data/stage_5_data/')

# The zip extracts as stage_5_data/cora etc — fix path if needed
import shutil
extracted = 'data/stage_5_data/stage_5_data'
if os.path.exists(extracted):
    for ds in ['cora', 'citeseer', 'pubmed']:
        shutil.move(f'{extracted}/{ds}', f'data/stage_5_data/{ds}')
    shutil.rmtree(extracted)

os.makedirs('result/stage_5_result', exist_ok=True)
print('Data ready:', os.listdir('data/stage_5_data'))

Saving stage_5_data.zip to stage_5_data.zip
Data ready: ['pubmed', '__MACOSX', 'cora', 'citeseer']


## 1 — Add repo root to path (replaces PyCharm Sources Root)

In [3]:
import sys
# This makes  `from code.base_class...`  and  `from local_code...`  resolve
# exactly the same way they do in PyCharm
sys.path.insert(0, '/content/ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template')
print('Path set.')

Path set.


## 2 — Run on Cora (5-3)

In [4]:
%cd /content/ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template

/content/ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template


In [5]:
import sys, types

TEMPLATE = '/content/ECS-170-Group-project/ECS170_Spring_2026_Source_Code_Template'

# Register code package directly in sys.modules — bypasses the built-in conflict
code_pkg = types.ModuleType('code')
code_pkg.__path__ = [f'{TEMPLATE}/code']
sys.modules['code'] = code_pkg

bc_pkg = types.ModuleType('code.base_class')
bc_pkg.__path__ = [f'{TEMPLATE}/code/base_class']
sys.modules['code.base_class'] = bc_pkg

ds_mod = types.ModuleType('code.base_class.dataset')
class dataset:
    data = None
    dataset_name = None
    dataset_source_folder_path = None
    def __init__(self, dName=None, dDescription=None):
        self.dataset_name = dName
ds_mod.dataset = dataset
sys.modules['code.base_class.dataset'] = ds_mod

mt_mod = types.ModuleType('code.base_class.method')
class method:
    def __init__(self, mName=None, mDescription=None):
        self.method_name = mName
        self.method_description = mDescription
mt_mod.method = method
sys.modules['code.base_class.method'] = mt_mod

sys.path.insert(0, TEMPLATE)
print('Done.')

Done.


In [7]:
print(metrics.keys())

NameError: name 'metrics' is not defined

In [11]:
import os
from local_code.stage_5_code.Dataset_Loader_Node_Classification import Dataset_Loader
from local_code.stage_5_code.Method_GCN_Node_Classification     import Method_GCN_Node_Classification
from local_code.stage_5_code.Evaluate_Metrics                   import Evaluate_Metrics

DATA_ROOT     = 'data/stage_5_data'
RESULT_FOLDER = 'result/stage_5_result'

def run_dataset(dataset_name):
    loader = Dataset_Loader(dName=dataset_name, dDescription='')
    loader.dataset_name               = dataset_name
    loader.dataset_source_folder_path = os.path.join(DATA_ROOT, dataset_name)

    loaded         = loader.load()
    graph          = loaded['graph']
    train_test_val = loaded['train_test_val']

    gcn = Method_GCN_Node_Classification('GCN', '2-layer GCN node classifier')
    gcn.dataset_name  = dataset_name
    gcn.result_folder = RESULT_FOLDER
    gcn.data = {
        'X':         graph['X'],
        'y':         graph['y'],
        'utility':   graph['utility'],
        'idx_train': train_test_val['idx_train'],
        'idx_val':   train_test_val['idx_val'],
        'idx_test':  train_test_val['idx_test'],
    }

    result = gcn.run()
    pred_y, true_y = gcn.run()

    metrics = Evaluate_Metrics().evaluate(pred_y, true_y)
    print('\n--- {} Results ---'.format(dataset_name.upper()))
    print('Accuracy    :', round(metrics['accuracy'],              4))
    print('Macro F1    :', round(metrics['macro_f1'],              4))
    print('Weighted F1 :', round(metrics['weighted_f1'],           4))
    print(metrics['classification_report'])
    return metrics

cora_metrics = run_dataset('cora')

Loading cora dataset...
method:GCN, dataset:cora
--start training...
Epoch:   0 | Train Loss: 1.9674 Acc: 0.5286 | Val Loss: 1.8034 Acc: 0.3233
Epoch:  20 | Train Loss: 0.0219 Acc: 1.0000 | Val Loss: 0.6528 Acc: 0.8000
Epoch:  40 | Train Loss: 0.0119 Acc: 1.0000 | Val Loss: 0.6152 Acc: 0.8100
Epoch:  60 | Train Loss: 0.0134 Acc: 1.0000 | Val Loss: 0.5833 Acc: 0.8033
Epoch:  80 | Train Loss: 0.0172 Acc: 1.0000 | Val Loss: 0.5632 Acc: 0.8167
Epoch: 100 | Train Loss: 0.0137 Acc: 1.0000 | Val Loss: 0.5751 Acc: 0.8067
Epoch: 120 | Train Loss: 0.0164 Acc: 1.0000 | Val Loss: 0.5682 Acc: 0.8167
Epoch: 140 | Train Loss: 0.0104 Acc: 1.0000 | Val Loss: 0.5773 Acc: 0.8033
Epoch: 160 | Train Loss: 0.0102 Acc: 1.0000 | Val Loss: 0.5844 Acc: 0.8000
Epoch: 180 | Train Loss: 0.0178 Acc: 1.0000 | Val Loss: 0.5749 Acc: 0.8033
--training done, time: 1.1s
Curve saved to: result/stage_5_result/learning_curve_cora.png
method:GCN, dataset:cora
--start training...
Epoch:   0 | Train Loss: 1.9337 Acc: 0.5571 | 

## 3 — Run on Citeseer (5-4)

In [12]:
citeseer_metrics = run_dataset('citeseer')

Loading citeseer dataset...
method:GCN, dataset:citeseer
--start training...
Epoch:   0 | Train Loss: 1.7859 Acc: 0.7417 | Val Loss: 1.9441 Acc: 0.1967
Epoch:  20 | Train Loss: 0.0055 Acc: 1.0000 | Val Loss: 2.0866 Acc: 0.4133
Epoch:  40 | Train Loss: 0.0071 Acc: 1.0000 | Val Loss: 1.6142 Acc: 0.4900
Epoch:  60 | Train Loss: 0.0074 Acc: 1.0000 | Val Loss: 1.6790 Acc: 0.4700
Epoch:  80 | Train Loss: 0.0122 Acc: 1.0000 | Val Loss: 1.6623 Acc: 0.4867
Epoch: 100 | Train Loss: 0.0055 Acc: 1.0000 | Val Loss: 1.6994 Acc: 0.4700
Epoch: 120 | Train Loss: 0.0078 Acc: 1.0000 | Val Loss: 1.5608 Acc: 0.4867
Epoch: 140 | Train Loss: 0.0080 Acc: 1.0000 | Val Loss: 1.6617 Acc: 0.4967
Epoch: 160 | Train Loss: 0.0063 Acc: 1.0000 | Val Loss: 1.5920 Acc: 0.4833
Epoch: 180 | Train Loss: 0.0051 Acc: 1.0000 | Val Loss: 1.5595 Acc: 0.5133
--training done, time: 1.2s
Curve saved to: result/stage_5_result/learning_curve_citeseer.png
method:GCN, dataset:citeseer
--start training...
Epoch:   0 | Train Loss: 1.814

## 4 — Run on Pubmed (5-4)

In [13]:
pubmed_metrics = run_dataset('pubmed')

Loading pubmed dataset...
method:GCN, dataset:pubmed
--start training...
Epoch:   0 | Train Loss: 1.0962 Acc: 0.7333 | Val Loss: 1.0807 Acc: 0.6000
Epoch:  20 | Train Loss: 0.2745 Acc: 0.9667 | Val Loss: 0.5965 Acc: 0.7833
Epoch:  40 | Train Loss: 0.0863 Acc: 1.0000 | Val Loss: 0.5799 Acc: 0.7767
Epoch:  60 | Train Loss: 0.0590 Acc: 1.0000 | Val Loss: 0.5837 Acc: 0.7667
Epoch:  80 | Train Loss: 0.0604 Acc: 1.0000 | Val Loss: 0.5918 Acc: 0.7800
Epoch: 100 | Train Loss: 0.0512 Acc: 1.0000 | Val Loss: 0.5906 Acc: 0.7667
Epoch: 120 | Train Loss: 0.0467 Acc: 1.0000 | Val Loss: 0.5941 Acc: 0.7767
Epoch: 140 | Train Loss: 0.0471 Acc: 1.0000 | Val Loss: 0.5996 Acc: 0.7667
Epoch: 160 | Train Loss: 0.0477 Acc: 1.0000 | Val Loss: 0.6073 Acc: 0.7733
Epoch: 180 | Train Loss: 0.0347 Acc: 1.0000 | Val Loss: 0.6083 Acc: 0.7800
--training done, time: 1.7s
Curve saved to: result/stage_5_result/learning_curve_pubmed.png
method:GCN, dataset:pubmed
--start training...
Epoch:   0 | Train Loss: 1.0967 Acc: 0

## 5 — Summary table

In [16]:
results = {'cora': cora_metrics, 'citeseer': citeseer_metrics, 'pubmed': pubmed_metrics}
print('\n{:<12} {:>10} {:>10} {:>12}'.format('Dataset','Accuracy','Macro F1','Weighted F1'))
print('-'*48)
for name, r in results.items():
    print('{:<12} {:>10.4f} {:>10.4f} {:>12.4f}'.format(
        name, r['accuracy'], r['macro_f1'], r['weighted_f1']))


Dataset        Accuracy   Macro F1  Weighted F1
------------------------------------------------
cora             0.8270     0.8098       0.8293
citeseer         0.7050     0.5565       0.6772
pubmed           0.7860     0.7819       0.7861


## 6 — Download learning curve images

In [17]:
from google.colab import files
import glob
for f in glob.glob('result/stage_5_result/*.png'):
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7 — Push results to GitHub

SyntaxError: invalid syntax (2741859652.py, line 1)

In [18]:
!git add result/stage_5_result/
!git commit -m "Stage 5: GCN results and learning curves"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@d029f42c1e6e.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
